# SparkOCR-VLM on Databricks Free Edition

Run this notebook inside a Databricks Free workspace. No cluster config needed.

In [ ]:
%pip install -q sparkocr-vlm
dbutils.library.restartPython()

In [ ]:
import os
# Best practice: use a secret scope.
try:
    os.environ['OPENROUTER_API_KEY'] = dbutils.secrets.get(scope='sparkocr', key='openrouter')
except Exception:
    # Demo fallback — paste your key here for ad-hoc runs (never commit).
    os.environ['OPENROUTER_API_KEY'] = 'sk-or-v1-...'

## Upload sample PDFs to a Volume first

In the Databricks UI: Catalog → workspace → default → Create Volume → `sparkocr_demo`. Then drag-drop your PDFs into `/Volumes/workspace/default/sparkocr_demo/input/`.

In [ ]:
INPUT  = '/Volumes/workspace/default/sparkocr_demo/input/'
OUTPUT = '/Volumes/workspace/default/sparkocr_demo/silver/'

dbutils.fs.ls(INPUT)

In [ ]:
from sparkocr_vlm import OCRPipeline

pipeline = OCRPipeline(
    backend='openrouter',
    model='deepseek-ai/DeepSeek-OCR-v2:free',
    input_path=INPUT,
    output_path=OUTPUT,
    batch_size=4,
    max_cost_usd=0.50,
)
silver = pipeline.run(spark)
display(silver)

## Query the silver table

In [ ]:
spark.read.format('delta').load(OUTPUT).createOrReplaceTempView('ocr_silver')

In [ ]:
%sql
SELECT filename, page_num, doc_type, cost_usd
FROM ocr_silver
ORDER BY cost_usd DESC
LIMIT 20